Run the following cell, which will set a Spark configuration variable that disables caching. Turning caching off makes the effect of the optimiations more apparent

In [0]:
%python
spark.conf.set('spark.databricks.io.cache.enabled', 'false')

**Process & Write IoT data**

Let's generate some fake IoT data. This first time around, we are going to generate 2500 rows

In [0]:
spark.range(0,2500).show(5)

In [0]:
## hash(id) produce un entero pseudoaleatorio pero determinístico para cada id.

spark.range(0,2500).select('id', hash('id')).show(5)

In [0]:
from pyspark.sql.functions import *
spark.range(2).select('id',rand().alias('value')).show()
## Genera un número aleatorio entre 0 y 1 para cada fila.

In [0]:
from pyspark.sql.functions import *

df = (spark
      .range(0, 2500)
      .select(
          hash('id').alias('id'),
          rand().alias('value'),
          from_unixtime(lit(1701692381 + col('id'))).alias('time')
      ))

display(df)

Now we'll write the data to a table partitioned by id, which will result in every row being written to a separate file. 2500 rows will take a long time to write in this fashion. Note how long it takes to generate the table. This id has a high cardinality, so it takes aprox. 55s. It generates 2500 files and metadata is stored in the metastore.

In [0]:
(df
 .write
 .mode('overwrite')
 .option('overwriteSchema', 'true')
 .partitionBy('time')
 .saveAsTable('iot_data')
)

**Query the Tabke**

Run the two queries against the table we just wrote. Note the time taken to execute each query. The first reading a single partition lasts 2s.

In [0]:
%sql
SELECT * FROM iot_data WHERE id = 519220707

In [0]:
%sql
SELECT avg(value) FROM iot_data WHERE time >= "2023-12-04 12:19:00" and time <= "2023-12-04 13:01:20"